In [0]:
dbutils.widgets.text("catalog", "tpch")

catalog = dbutils.widgets.get("catalog")
print(f"Selected catalog: {catalog}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
# Create volume for checkpoints if not exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")
# print(f"Volume '{catalog}.bronze.checkpoints' created or already exists.")

# List all tables in samples.tpch
_tpch_tables = spark.sql("SHOW TABLES IN samples.tpch").collect()
_table_names = [row.tableName for row in _tpch_tables]
# print(f"\nFound {len(_table_names)} tables in samples.tpch:")

# Create checkpoint folder for each table if not exists
for _name in _table_names:
    _checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints/{_name}"
    dbutils.fs.mkdirs(_checkpoint_path)
    # print(f"  - {_name} -> {_checkpoint_path}")

# Create volume and checkpoint folders for silver
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.silver.checkpoints")
for _name in _table_names:
    _checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{_name}"
    dbutils.fs.mkdirs(_checkpoint_path)


In [0]:
import io
import contextlib
import re
from pyspark.sql.functions import current_timestamp, lit, col

def _get_job_run_id():
    """Get the current Databricks job run ID, or 'interactive' if not in a job."""
    try:
        run_id = dbutils.notebook.getContext().currentRunId()
        if run_id.isDefined():
            return str(run_id.get())
    except Exception:
        pass
    return "interactive"

def _extract_table_name(df):
    """Try to extract the table name from the DataFrame's explain output."""
    try:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            df.explain(extended=True)
        explain_str = buf.getvalue()

        # UnresolvedRelation [catalog, schema, table]
        match = re.search(r'UnresolvedRelation\s*\[([^\]]+)\]', explain_str)
        if match:
            parts = [p.strip().strip("'") for p in match.group(1).split(',')]
            return '.'.join(parts)

        # SubqueryAlias catalog.schema.table
        match = re.search(r'SubqueryAlias\s+(\S+)', explain_str)
        if match:
            return match.group(1)
    except Exception:
        pass
    return None

def add_meta_columns(df):
    """Add metadata columns to a DataFrame and return it.

    Automatically detects the data source (table name or file path) and adds:
    - _data_source: source table name or file path (via _metadata.file_path)
    - _ingested_at: current timestamp
    - _job_run_id: Databricks job run ID (or 'interactive')

    Args:
        df: Source Spark DataFrame (from spark.table, spark.read, etc.).

    Returns:
        DataFrame with metadata columns appended.
    """
    job_run_id = _get_job_run_id()
    table_name = _extract_table_name(df)

    if table_name:
        source_col = lit(table_name)
    else:
        source_col = col("_metadata.file_path")

    return df.withColumns({
        "_data_source": source_col,
        "_ingested_at": current_timestamp(),
        "_job_run_id": lit(job_run_id)
    })